In [1]:
import threading
import time

In [2]:
threading.active_count()

9

In [3]:
threading.enumerate()

[<_MainThread(MainThread, started 8294030720)>,
 <Thread(IOPub, started daemon 6148255744)>,
 <Heartbeat(Heartbeat, started daemon 6165082112)>,
 <Thread(Thread-2 (_watch_pipe_fd), started daemon 6183055360)>,
 <Thread(Thread-3 (_watch_pipe_fd), started daemon 6199881728)>,
 <ControlThread(Control, started daemon 6216708096)>,
 <ShellChannelThread(Shell channel, started daemon 6233534464)>,
 <HistorySavingThread(IPythonHistorySavingThread, started 6250360832)>,
 <ParentPollerUnix(Thread-1, started daemon 6267760640)>]

In [4]:
threading.current_thread(), threading.currentThread()

/var/folders/j3/m947cq0j0fx8t0rrdzw5kmb00000gp/T/ipykernel_72435/633031836.py:1: DeprecationWarning: currentThread() is deprecated, use current_thread() instead
  threading.current_thread(), threading.currentThread()


(<_MainThread(MainThread, started 8294030720)>,
 <_MainThread(MainThread, started 8294030720)>)

In [6]:
th = threading.current_thread()

In [7]:
th.name, th.native_id, th.ident, th.is_alive(), th.daemon

('MainThread', 534164250, 8294030720, True, False)

In [8]:
threading.get_ident(), threading.get_native_id()

(8294030720, 534164250)

In [9]:
class DataProcessor(threading.Thread):
    def __init__(self, n):
        super().__init__()
        self.n = n
        self.result = 0

    def run(self):
        print(f"started {self.name=}, {self.ident=}, {self.daemon=}")

        for i in range(self.n):
            print(f"run: {i=}, {self.result=}")
            self.result += i
            time.sleep(1)
        
        print(f"finished {self.name=}, {self.ident=}, {self.daemon=}")

In [10]:
data = DataProcessor(10)
data.start()
print("--- after start", data.is_alive(), data.result)

started self.name='Thread-5', self.ident=6284587008, self.daemon=False--- after start True 0

run: i=0, self.result=0
run: i=1, self.result=0
run: i=2, self.result=1


In [11]:
print("--- after start", data.is_alive(), data.result)

--- after start True 3
run: i=3, self.result=3
run: i=4, self.result=6
run: i=5, self.result=10


In [12]:
print("--- after start", data.is_alive(), data.result)

--- after start True 15
run: i=6, self.result=15
run: i=7, self.result=21
run: i=8, self.result=28
run: i=9, self.result=36
finished self.name='Thread-5', self.ident=6284587008, self.daemon=False


In [13]:
print("--- after start", data.is_alive(), data.result)

--- after start False 45


In [15]:
data = DataProcessor(10)
data.start()
data.join(5)
print("--- after start", data.is_alive(), data.result)

started self.name='Thread-7', self.ident=6284587008, self.daemon=False
run: i=0, self.result=0
run: i=1, self.result=0
run: i=2, self.result=1
run: i=3, self.result=3
run: i=4, self.result=6
--- after start True 10
run: i=5, self.result=10
run: i=6, self.result=15


In [16]:
print("--- after start", data.is_alive(), data.result)

--- after start True 21
run: i=7, self.result=21
run: i=8, self.result=28
run: i=9, self.result=36
finished self.name='Thread-7', self.ident=6284587008, self.daemon=False


In [17]:
print("--- after start", data.is_alive(), data.result)

--- after start False 45


In [18]:
%%time

N = 10 ** 9

def counter(a, b):
    while a < b:
        a += 1


counter(0, N)

CPU times: user 8.52 s, sys: 107 ms, total: 8.63 s
Wall time: 8.7 s


In [23]:
%%time

# [0, 100] == [0, 10] [10, 20] ... [90, 100]


N = 10 ** 9
N_THREADS = 10
STEP = N // N_THREADS


def counter(a, b):
    print(f"--- started {threading.current_thread().name=}, {a=}, {b=}")
    while a < b:
        a += 1

    print(f"--- finished {threading.current_thread().name=}")


threads = [
    threading.Thread(
        target=counter,
        args=(i * STEP, (i + 1) * STEP),
        name=f"th_counter_{i}",
    )
    for i in range(N_THREADS)
]


for th in threads:
    th.start()


for th in threads:
    th.join()

--- started threading.current_thread().name='th_counter_0', a=0, b=100000000--- started threading.current_thread().name='th_counter_1', a=100000000, b=200000000

--- started threading.current_thread().name='th_counter_2', a=200000000, b=300000000
--- started threading.current_thread().name='th_counter_3', a=300000000, b=400000000
--- started threading.current_thread().name='th_counter_4', a=400000000, b=500000000
--- started threading.current_thread().name='th_counter_5', a=500000000, b=600000000
--- started threading.current_thread().name='th_counter_6', a=600000000, b=700000000
--- started threading.current_thread().name='th_counter_7', a=700000000, b=800000000
--- started threading.current_thread().name='th_counter_8', a=800000000, b=900000000
--- started threading.current_thread().name='th_counter_9', a=900000000, b=1000000000
--- finished threading.current_thread().name='th_counter_1'
--- finished threading.current_thread().name='th_counter_4'
--- finished threading.current_thread

In [24]:
!cat counter.py

import threading
import time


N = 10 ** 9  # [0-1, 1-2, 2-3]
N_THREADS = 10
N_JOBS = N // N_THREADS


def counter(a, b):
    print(f"counter[{a}, {b}], name={threading.current_thread().name} -- started")
    while a < b:
        a += 1
    print(f"counter[{a}, {b}], name={threading.current_thread().name} -- finished")



def run():
    threads = [
        threading.Thread(
            target=counter,
            args=(i * N_JOBS, (i + 1) * N_JOBS),
            name=f"th_counter_{i}",
        )
        for i in range(N_THREADS)
    ]

    for th in threads:
        th.start()

    for th in threads:
        th.join()


if __name__ == "__main__":
    t1 = time.time()
    run()
    t2 = time.time()
    print(f"time={t2 - t1}")


In [25]:
!python counter.py

counter[0, 100000000], name=th_counter_0 -- started
counter[100000000, 200000000], name=th_counter_1 -- started
counter[200000000, 300000000], name=th_counter_2 -- started
counter[300000000, 400000000], name=th_counter_3 -- started
counter[400000000, 500000000], name=th_counter_4 -- started
counter[500000000, 600000000], name=th_counter_5 -- started
counter[600000000, 700000000], name=th_counter_6 -- started
counter[700000000, 800000000], name=th_counter_7 -- started
counter[800000000, 900000000], name=th_counter_8 -- started
counter[900000000, 1000000000], name=th_counter_9 -- started
counter[500000000, 500000000], name=th_counter_4 -- finished
counter[200000000, 200000000], name=th_counter_1 -- finished
counter[700000000, 700000000], name=th_counter_6 -- finished
counter[800000000, 800000000], name=th_counter_7 -- finished
counter[300000000, 300000000], name=th_counter_2 -- finished
counter[400000000, 400000000], name=th_counter_3 -- finished
counter[600000000, 600000000], name=th_co

In [26]:
!python3.14t counter.py

counter[0, 100000000], name=th_counter_0 -- started
counter[100000000, 200000000], name=th_counter_1 -- started
counter[200000000, 300000000], name=th_counter_2 -- started
counter[300000000, 400000000], name=th_counter_3 -- started
counter[400000000, 500000000], name=th_counter_4 -- started
counter[500000000, 600000000], name=th_counter_5 -- started
counter[600000000, 700000000], name=th_counter_6 -- started
counter[700000000, 800000000], name=th_counter_7 -- started
counter[800000000, 900000000], name=th_counter_8 -- started
counter[900000000, 1000000000], name=th_counter_9 -- started
counter[100000000, 100000000], name=th_counter_0 -- finished
counter[1000000000, 1000000000], name=th_counter_9 -- finished
counter[300000000, 300000000], name=th_counter_2 -- finished
counter[200000000, 200000000], name=th_counter_1 -- finished
counter[900000000, 900000000], name=th_counter_8 -- finished
counter[800000000, 800000000], name=th_counter_7 -- finished
counter[500000000, 500000000], name=th_

In [31]:
%%time


N = 10 ** 8
N_THREADS = 10
STEP = N // N_THREADS


class PyObject:
    def __init__(self):
        self.ref_counter = 0


obj = PyObject()


def counter(a, b):
    print(f"--- started {threading.current_thread().name=}, {a=}, {b=}")
    while a < b:
        a += 1

        tmp = obj.ref_counter  # th1, th2, th5
        x = min(tmp, 12)       # th1

        obj.ref_counter = tmp + 1  # th1: 1, th2:1

    print(f"--- finished {threading.current_thread().name=}")


threads = [
    threading.Thread(
        target=counter,
        args=(i * STEP, (i + 1) * STEP),
        name=f"th_counter_{i}",
    )
    for i in range(N_THREADS)
]

print(f"before {obj.ref_counter=}")


for th in threads:
    th.start()


for th in threads:
    th.join()

print(f"after {obj.ref_counter=}, {obj.ref_counter == N=}")

before obj.ref_counter=0
--- started threading.current_thread().name='th_counter_0', a=0, b=10000000
--- started threading.current_thread().name='th_counter_1', a=10000000, b=20000000
--- started threading.current_thread().name='th_counter_2', a=20000000, b=30000000
--- started threading.current_thread().name='th_counter_3', a=30000000, b=40000000
--- started threading.current_thread().name='th_counter_4', a=40000000, b=50000000
--- started threading.current_thread().name='th_counter_5', a=50000000, b=60000000
--- started threading.current_thread().name='th_counter_6', a=60000000, b=70000000--- started threading.current_thread().name='th_counter_7', a=70000000, b=80000000

--- started threading.current_thread().name='th_counter_8', a=80000000, b=90000000
--- started threading.current_thread().name='th_counter_9', a=90000000, b=100000000
--- finished threading.current_thread().name='th_counter_4'
--- finished threading.current_thread().name='th_counter_2'--- finished threading.current_t

In [36]:
%%time


N = 10 ** 8
N_THREADS = 10
STEP = N // N_THREADS


class PyObject:
    def __init__(self):
        self.ref_counter = 0



def counter(a, b, lock, obj):
    print(f"--- started {threading.current_thread().name=}, {a=}, {b=}")
    while a < b:
        a += 1

        with lock:
            tmp = obj.ref_counter  # th1, th2, th5
            x = min(tmp, 12)       # th1
    
            obj.ref_counter = tmp + 1  # th1: 1, th2:1

    print(f"--- finished {threading.current_thread().name=}")



def run():
    obj = PyObject()
    lock = threading.Lock()

    threads = [
        threading.Thread(
            target=counter,
            args=(i * STEP, (i + 1) * STEP, lock, obj),
            name=f"th_counter_{i}",
        )
        for i in range(N_THREADS)
    ]
    
    print(f"before {obj.ref_counter=}")
    
    
    for th in threads:
        th.start()
    
    
    for th in threads:
        th.join()
    
    print(f"after {obj.ref_counter=}, {obj.ref_counter == N=}")


run()

before obj.ref_counter=0
--- started threading.current_thread().name='th_counter_0', a=0, b=10000000
--- started threading.current_thread().name='th_counter_1', a=10000000, b=20000000
--- started threading.current_thread().name='th_counter_2', a=20000000, b=30000000
--- started threading.current_thread().name='th_counter_3', a=30000000, b=40000000
--- started threading.current_thread().name='th_counter_4', a=40000000, b=50000000
--- started threading.current_thread().name='th_counter_5', a=50000000, b=60000000
--- started threading.current_thread().name='th_counter_6', a=60000000, b=70000000
--- started threading.current_thread().name='th_counter_7', a=70000000, b=80000000
--- started threading.current_thread().name='th_counter_8', a=80000000, b=90000000
--- started threading.current_thread().name='th_counter_9', a=90000000, b=100000000
--- finished threading.current_thread().name='th_counter_1'
--- finished threading.current_thread().name='th_counter_9'
--- finished threading.current_

In [37]:
URL = "https://www.python.org/"

In [38]:
import requests

In [39]:
resp = requests.get(URL)

In [44]:
resp.status_code

200

In [50]:
%%time


N = 20
N_THREADS = 5
STEP = N // N_THREADS

URL = "https://www.python.org/"
URLS = [URL] * N


def fetch_urls(urls):
    print(f"--- started {threading.current_thread().name=}")
    t1 = time.time()
    for url in urls:
        resp = requests.get(url)
        #time.sleep(1)
        print(resp.status_code)

    t2 = time.time()
    print(f"--- finished {threading.current_thread().name=}", t2 - t1)


def run():
    threads = [
        threading.Thread(
            target=fetch_urls,
            args=(URLS[i * STEP : (i + 1) * STEP],),
            name=f"th_counter_{i}",
        )
        for i in range(N_THREADS)
    ]
    
    for th in threads:
        th.start()
    
    for th in threads:
        th.join()


run()

--- started threading.current_thread().name='th_counter_0'
--- started threading.current_thread().name='th_counter_1'
--- started threading.current_thread().name='th_counter_2'
--- started threading.current_thread().name='th_counter_3'
--- started threading.current_thread().name='th_counter_4'
200
200
200
200
200
200
200
200
200
200
200
200
--- finished threading.current_thread().name='th_counter_1' 1.2248859405517578
200
200
--- finished threading.current_thread().name='th_counter_4' 1.343980073928833
200
--- finished threading.current_thread().name='th_counter_2' 1.3731591701507568
200
200
--- finished threading.current_thread().name='th_counter_3' 1.5477619171142578
200
200
200
--- finished threading.current_thread().name='th_counter_0' 2.230504274368286
CPU times: user 664 ms, sys: 68.9 ms, total: 733 ms
Wall time: 2.24 s


In [51]:
import queue

In [52]:
%%time


N = 20
N_THREADS = 5
STEP = N // N_THREADS

URL = "https://www.python.org/"
URLS = [URL] * N


def fetch_urls(que):
    print(f"--- started {threading.current_thread().name=}")
    t1 = time.time()
    cnt = 0
    
    while True:
        url = que.get()
        if url is None:
            que.put(url)
            break

        cnt += 1
        resp = requests.get(url)
        #time.sleep(1)
        print(resp.status_code)

    t2 = time.time()
    print(f"--- finished {threading.current_thread().name=}", t2 - t1, cnt)


def run():
    que = queue.Queue()
    for url in URLS:
        que.put(url)
    que.put(None)

    threads = [
        threading.Thread(
            target=fetch_urls,
            args=(que,),
            name=f"th_counter_{i}",
        )
        for i in range(N_THREADS)
    ]
    
    for th in threads:
        th.start()
    
    for th in threads:
        th.join()


run()

--- started threading.current_thread().name='th_counter_0'--- started threading.current_thread().name='th_counter_1'

--- started threading.current_thread().name='th_counter_2'
--- started threading.current_thread().name='th_counter_3'
--- started threading.current_thread().name='th_counter_4'
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
--- finished threading.current_thread().name='th_counter_1' 1.6234328746795654 4
200
--- finished threading.current_thread().name='th_counter_0' 1.778709888458252 3
200
--- finished threading.current_thread().name='th_counter_2' 1.80180025100708 5
200
--- finished threading.current_thread().name='th_counter_4' 1.862595796585083 5
200
--- finished threading.current_thread().name='th_counter_3' 1.9170968532562256 3
CPU times: user 817 ms, sys: 80.7 ms, total: 898 ms
Wall time: 1.93 s
